# 00 — Kiểm tra môi trường dự án BTL-K03

Notebook này **chỉ kiểm tra**, không cài đặt hoặc nâng cấp package.

Các mục kiểm tra:

1. Python kernel có đúng venv `D:\DL\.venv` không.
2. Phiên bản Python, hệ điều hành và các thư viện cần thiết.
3. PyTorch, CUDA và GPU.
4. Cấu trúc thư mục dự án và quyền ghi vào các thư mục đầu ra.
5. Sự tồn tại, header, dung lượng và SHA-256 của ba file Diginetica.

> Không chạy `pip install` trong notebook này. Danh sách thiếu được in ở cell tổng kết cuối cùng.

In [ ]:
from __future__ import annotations

import hashlib
import importlib
import importlib.util
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "BTL-K03-tong-quan-de-tai.md").is_file():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy project root. Hãy mở notebook từ bên trong dự án DA-3."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "diginetica"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"PROJECT_ROOT       : {PROJECT_ROOT}")
print(f"RAW_DATA_DIR       : {RAW_DATA_DIR}")
print(f"INTERIM_DATA_DIR   : {INTERIM_DATA_DIR}")
print(f"PROCESSED_DATA_DIR : {PROCESSED_DATA_DIR}")
print(f"RUNS_DIR           : {RUNS_DIR}")
print(f"RESULTS_DIR        : {RESULTS_DIR}")

## 1. Kiểm tra Python kernel

Venv dự kiến của dự án hiện tại là `D:\DL\.venv`. Nếu `sys.executable` không trỏ đến venv này, cần chọn lại kernel trong VS Code/Jupyter trước khi tiếp tục.

In [ ]:
EXPECTED_VENV = Path(r"D:\DL\.venv").resolve()
EXPECTED_PYTHON = EXPECTED_VENV / "Scripts" / "python.exe"
CURRENT_PYTHON = Path(sys.executable).resolve()
USING_EXPECTED_VENV = CURRENT_PYTHON == EXPECTED_PYTHON

print(f"Python version     : {platform.python_version()}")
print(f"Python executable  : {CURRENT_PYTHON}")
print(f"Expected executable: {EXPECTED_PYTHON}")
print(f"Đúng venv          : {USING_EXPECTED_VENV}")
print(f"Operating system   : {platform.platform()}")

if not USING_EXPECTED_VENV:
    print("\nCẢNH BÁO: Notebook đang không dùng D:\\DL\\.venv.")
    print("Hãy chọn kernel 'dl-venv' hoặc interpreter D:\\DL\\.venv\\Scripts\\python.exe.")

## 2. Kiểm tra thư viện

Các package được chia theo giai đoạn:

- `EDA`: cần ngay cho khám phá và tiền xử lý dữ liệu.
- `TRAINING`: cần khi huấn luyện GRU4Rec, NARM và SASRec.
- `TOOLING`: cần để chạy notebook và lưu định dạng dữ liệu.
- `RECBOLE_DEP`: dependency được RecBole công bố; chưa nhất thiết phải cài riêng vì trình cài RecBole có thể xử lý.

Cột `import_ok` mới là kiểm tra quan trọng nhất. Có trường hợp module import được nhưng metadata của `pip` không đầy đủ.

In [ ]:
DEPENDENCIES = [
    # distribution, import name, stage, purpose
    ("numpy", "numpy", "EDA", "Tính toán mảng"),
    ("pandas", "pandas", "EDA", "Đọc và xử lý bảng dữ liệu"),
    ("scipy", "scipy", "EDA", "Spearman và kiểm định thống kê"),
    ("scikit-learn", "sklearn", "EDA", "t-SNE, PCA và cosine"),
    ("matplotlib", "matplotlib", "EDA", "Vẽ biểu đồ"),
    ("seaborn", "seaborn", "EDA", "Vẽ biểu đồ thống kê"),
    ("pyarrow", "pyarrow", "TOOLING", "Đọc/ghi Parquet"),
    ("PyYAML", "yaml", "TOOLING", "Đọc cấu hình YAML"),
    ("jupyter", "jupyter", "TOOLING", "Chạy Jupyter"),
    ("notebook", "notebook", "TOOLING", "Giao diện notebook"),
    ("ipykernel", "ipykernel", "TOOLING", "Python kernel"),
    ("nbformat", "nbformat", "TOOLING", "Định dạng notebook"),
    ("tqdm", "tqdm", "TOOLING", "Progress bar"),
    ("torch", "torch", "TRAINING", "Huấn luyện deep learning"),
    ("recbole", "recbole", "TRAINING", "GRU4Rec, NARM và SASRec"),
    ("tensorboard", "tensorboard", "RECBOLE_DEP", "Theo dõi quá trình huấn luyện"),
    ("hyperopt", "hyperopt", "RECBOLE_DEP", "Tuning tham số"),
    ("colorlog", "colorlog", "RECBOLE_DEP", "Log có màu"),
    ("colorama", "colorama", "RECBOLE_DEP", "Màu terminal trên Windows"),
    ("thop", "thop", "RECBOLE_DEP", "Thống kê độ phức tạp mô hình"),
    ("ray", "ray", "RECBOLE_DEP", "Tuning song song"),
    ("tabulate", "tabulate", "RECBOLE_DEP", "In bảng"),
    ("plotly", "plotly", "RECBOLE_DEP", "Biểu đồ tương tác"),
    ("texttable", "texttable", "RECBOLE_DEP", "In bảng dạng text"),
    ("psutil", "psutil", "RECBOLE_DEP", "Theo dõi tài nguyên"),
]


def get_distribution_version(distribution: str, imported_module) -> str:
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return str(getattr(imported_module, "__version__", "unknown"))


dependency_results = []
for distribution, module_name, stage, purpose in DEPENDENCIES:
    module_available = importlib.util.find_spec(module_name) is not None
    imported_module = None
    import_error = None

    if module_available:
        try:
            imported_module = importlib.import_module(module_name)
        except Exception as exc:
            import_error = f"{type(exc).__name__}: {exc}"

    import_ok = module_available and import_error is None
    version = (
        get_distribution_version(distribution, imported_module)
        if import_ok
        else None
    )

    dependency_results.append(
        {
            "package": distribution,
            "module": module_name,
            "stage": stage,
            "import_ok": import_ok,
            "version": version,
            "purpose": purpose,
            "error": import_error,
        }
    )

header = f"{'PACKAGE':<18} {'STAGE':<12} {'STATUS':<10} {'VERSION':<16} PURPOSE"
print(header)
print("-" * len(header))
for row in dependency_results:
    status = "OK" if row["import_ok"] else "MISSING"
    version = row["version"] or "-"
    print(
        f"{row['package']:<18} {row['stage']:<12} "
        f"{status:<10} {version:<16} {row['purpose']}"
    )
    if row["error"]:
        print(f"  Import error: {row['error']}")

### Lưu ý về RecBole

Môi trường hiện dùng Python 3.12 và các phiên bản thư viện khá mới. RecBole 1.2.1 có một số dependency cũ hoặc bị giới hạn phiên bản, đặc biệt là Ray. Vì vậy **không cài RecBole ngay trong notebook này**. Trước bước huấn luyện cần chốt một tổ hợp phiên bản tương thích hoặc tạo một venv huấn luyện riêng nếu cần.

## 3. Kiểm tra PyTorch, CUDA và GPU

In [ ]:
torch_result = next(
    row for row in dependency_results if row["package"] == "torch"
)

if not torch_result["import_ok"]:
    print("PyTorch chưa import được; bỏ qua kiểm tra GPU.")
else:
    import torch

    print(f"PyTorch version : {torch.__version__}")
    print(f"CUDA build      : {torch.version.cuda}")
    print(f"CUDA available  : {torch.cuda.is_available()}")
    print(f"GPU count       : {torch.cuda.device_count()}")

    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        memory_gb = properties.total_memory / (1024**3)
        print(
            f"GPU {index}: {properties.name} | "
            f"VRAM={memory_gb:.2f} GB | capability={properties.major}.{properties.minor}"
        )

## 4. Kiểm tra cấu trúc dự án và dữ liệu raw

Cell này không thay đổi dữ liệu. SHA-256 được so sánh với ba file đã kiểm tra ngày 23/10/2026. Nếu sau này thay nguồn dữ liệu hợp lệ, hash có thể khác và cần xác minh lại thay vì kết luận ngay rằng file hỏng.

In [ ]:
EXPECTED_RAW_FILES = {
    "train-item-views.csv": {
        "header": "sessionId;userId;itemId;timeframe;eventdate",
        "sha256": "5A27DBD1A7D47867EF64AFF378DBED971131F0F6FCD86FEE8BC1C8CBED6F413F",
    },
    "product-categories.csv": {
        "header": "itemId;categoryId",
        "sha256": "E5B7D2CDA115F0EC3F5D5D571D247147A23A5EC1BF62492A2FEB6A90B68FAFA2",
    },
    "products.csv": {
        "header": "itemId;pricelog2;product.name.tokens",
        "sha256": "CFE9D17EBD07DBA16879A04FC556F223F758001F99A2E3BCCB71D8EDFFFA6CAC",
    },
}


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest().upper()


raw_file_results = []
for filename, expected in EXPECTED_RAW_FILES.items():
    path = RAW_DATA_DIR / filename
    exists = path.is_file()
    size_bytes = path.stat().st_size if exists else 0
    header = None
    digest = None

    if exists:
        with path.open("r", encoding="utf-8-sig", errors="replace") as stream:
            header = stream.readline().rstrip("\r\n")
        digest = sha256_file(path)

    raw_file_results.append(
        {
            "file": filename,
            "exists": exists,
            "size_mb": round(size_bytes / (1024**2), 2),
            "header_ok": header == expected["header"] if exists else False,
            "hash_ok": digest == expected["sha256"] if exists else False,
            "actual_header": header,
        }
    )

header = f"{'FILE':<27} {'EXISTS':<8} {'MB':>8} {'HEADER':<10} {'SHA256':<10}"
print(header)
print("-" * len(header))
for row in raw_file_results:
    print(
        f"{row['file']:<27} {str(row['exists']):<8} "
        f"{row['size_mb']:>8.2f} {str(row['header_ok']):<10} {str(row['hash_ok']):<10}"
    )
    if row["exists"] and not row["header_ok"]:
        print(f"  Header thực tế: {row['actual_header']}")

In [ ]:
PROJECT_DIRECTORIES = {
    "raw": RAW_DATA_DIR,
    "interim": INTERIM_DATA_DIR,
    "processed": PROCESSED_DATA_DIR,
    "runs": RUNS_DIR,
    "results": RESULTS_DIR,
}

print(f"{'DIRECTORY':<12} {'EXISTS':<8} {'WRITABLE':<10} PATH")
print("-" * 90)
for name, path in PROJECT_DIRECTORIES.items():
    exists = path.is_dir()
    writable = os.access(path, os.W_OK) if exists else False
    print(f"{name:<12} {str(exists):<8} {str(writable):<10} {path}")

print("\nLưu ý: notebook không thử ghi vào data/raw; cột WRITABLE chỉ phản ánh quyền hệ thống.")

## 5. Tổng kết

Cell cuối chỉ liệt kê vấn đề; không tự động sửa hoặc cài đặt gì.

In [ ]:
missing_by_stage = {}
for stage in ("EDA", "TOOLING", "TRAINING", "RECBOLE_DEP"):
    missing_by_stage[stage] = [
        row["package"]
        for row in dependency_results
        if row["stage"] == stage and not row["import_ok"]
    ]

invalid_raw_files = [
    row["file"]
    for row in raw_file_results
    if not (row["exists"] and row["header_ok"] and row["hash_ok"])
]

print("KẾT QUẢ KIỂM TRA")
print("=" * 60)
print(f"Đúng venv D:\\DL\\.venv : {USING_EXPECTED_VENV}")
print(f"Thiếu cho EDA           : {missing_by_stage['EDA'] or 'Không'}")
print(f"Thiếu tooling           : {missing_by_stage['TOOLING'] or 'Không'}")
print(f"Thiếu để huấn luyện     : {missing_by_stage['TRAINING'] or 'Không'}")
print(f"Thiếu dependency RecBole: {missing_by_stage['RECBOLE_DEP'] or 'Không'}")
print(f"File raw chưa hợp lệ    : {invalid_raw_files or 'Không'}")

eda_ready = (
    USING_EXPECTED_VENV
    and not missing_by_stage["EDA"]
    and not missing_by_stage["TOOLING"]
    and not invalid_raw_files
)
training_ready = (
    eda_ready
    and not missing_by_stage["TRAINING"]
    and not missing_by_stage["RECBOLE_DEP"]
)

print(f"\nSẵn sàng làm EDA       : {eda_ready}")
print(f"Sẵn sàng huấn luyện    : {training_ready}")

if eda_ready and not training_ready:
    print("\nCó thể bắt đầu EDA. Phần RecBole sẽ được xử lý riêng trước giai đoạn huấn luyện.")